In [ ]:
## Find out what GPU we got (and make sure we actually have one!)
!nvidia-smi

In [ ]:
## Download all the data! This may take a little while...

## Processed data
try:
    import google.colab
    IN_COLAB = True
    !mkdir ../data/
    !wget https://www.dropbox.com/s/la05h49y9ths7x3/autoseg_data.tar.gz?dl=0 -O /data/AllProcessedData.tar
    !tar -xf /data/AllProcessedData.tar -C /data/
    !rm /data/AllProcessedData.tar
except:
    IN_COLAB = False
    !wget https://www.dropbox.com/s/la05h49y9ths7x3/autoseg_data.tar.gz?dl=0 -O ./AllProcessedData.tar
    !tar -xf ./AllProcessedData.tar -C ./
    !rm AllProcessedData.tar

# utilities python file
if IN_COLAB:
    !wget https://www.dropbox.com/s/rng7h9mgkwaolt8/utils.py?dl=0 -O /content/utils.py
    !wget https://github.com/Christie-Scientific-Computing/HSST_Workshop/raw/refs/heads/main/setup/database.db -O /data/database.db
else:
    !wget https://www.dropbox.com/s/rng7h9mgkwaolt8/utils.py?dl=0 -O ./utils.py
    !wget https://github.com/Christie-Scientific-Computing/HSST_Workshop/raw/refs/heads/main/setup/database.db -O ./database.db

## 0. Install prerequisites

Here we install the pytorch flavour of the segmentation-models library, along with pydicom and pytorch-lightning

In [ ]:
if IN_COLAB:
    %pip install git+https://github.com/qubvel/segmentation_models.pytorch
    %pip install pytorch_lightning tqdm

## 0. Set up monitoring and enable matplotlib notebook interactivity

In [ ]:
%matplotlib inline
import tensorboard
%load_ext tensorboard

## 0. Load required libraries

In [ ]:
## 0. Load required libraries
import segmentation_models_pytorch as smp
from collections import defaultdict
import matplotlib.pyplot as plt
import pytorch_lightning as pl
import albumentations as A
from pathlib import Path
import numpy as np
import sqlite3
# import pydicom
import torch
import os
from utils import getFiles
from os.path import join
import pickle

## See if we're in colab...
if IN_COLAB:
    datapath = "/data/HnN_data/"
else:
    datapath = "/home/HnN_data/"  ## <---- You will need to change this if running locally

## 1. Query SQL Database


As before, check that the database exists, and try to connect to it

In [ ]:
if not os.path.isfile("/data/database.db"):
  print("Database doesn't exist")
else:
  con = sqlite3.connect("/data/database.db")
  print("Database found")

  # Create a cursor to execute commands
  cur = con.cursor()

In [ ]:
if cur:
  # Execute a query to verify which tables are present in the db, and their entry columns
  cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
  tables = [row[0] for row in cur.fetchall() if row[0] != 'sqlite_sequence']

  for table in tables:
      cur.execute(f"PRAGMA table_info({table})")
      columns = cur.fetchall()

      print(f"\n{table}")
      for col in columns:
          name, dtype = col[1], col[2]
          print(f"  {name:<25} {dtype:<10}")

 Filter db for H+N patients

In [ ]:
# Query DB for list of unique sites

# Select the relevant sites

# Filter the list of patients for the chosen sites



In [ ]:
# Repeat demographics visualisation on our H+N patient subset


In [ ]:
# Extract the filepaths for the patients we want to use


Organise the data

In [ ]:
# Split data into images and masks, training and test groups



# 2. Load the HN data

In [ ]:
## Get the training data
fnames = sorted(train_ims_paths)
train_ct_slices   = np.zeros((len(fnames), 256, 256), dtype=np.float32)
train_mask_slices = np.zeros((len(fnames), 256, 256), dtype=np.float32)

for fdx, fname in enumerate(fnames):
    train_ct_slices[fdx]   = np.load(train_ims_paths[fname])
    train_mask_slices[fdx] = np.load(train_masks_paths[fname])

# load pixel spacing
with open(join(datapath, "train", "spacings.pkl"), "rb") as f:
    train_pixel_sizes = pickle.load(f)


# Get the test data
fnames = sorted(test_ims_paths)
test_ct_slices   = np.zeros((len(fnames), 256, 256), dtype=np.float32)
test_mask_slices = np.zeros((len(fnames), 256, 256), dtype=np.float32)

for fdx, fname in enumerate(fnames):
    test_ct_slices[fdx]   = np.load(test_ims_paths[fname])
    test_mask_slices[fdx] = np.load(test_masks_paths[fname])

# load pixel spacing
with open(join(datapath, "test", "spacings.pkl"), "rb") as f:
    test_pixel_sizes = pickle.load(f)

print(train_ct_slices.shape)
print(train_mask_slices.shape)
print(test_ct_slices.shape)
print(test_mask_slices.shape)

# 3. Preprocessing

In [ ]:
# your code here ...

We have a slightly smaller dataset of HnN data. Therefore we don't need to sample the data as heavily.

In [ ]:
np.random.seed(1234)
subset_indices = np.random.permutation(window_levelled_slices_train.shape[0])

wl_slice_subset_train = window_levelled_slices_train[subset_indices[:400]]
mask_subset_train = train_mask_slices[subset_indices[:400]]
spacings_subset_train = np.array(list(train_pixel_sizes.values()))[subset_indices[:400]]

wl_slice_subset_val = window_levelled_slices_train[subset_indices[400:]]
mask_subset_val =  train_mask_slices[subset_indices[400:]]
spacings_subset_val = np.array(list(train_pixel_sizes.values()))[subset_indices[400:]]

np.random.seed()

In [ ]:
# add dataset code and dataloaders here ...

# 4. Creating the Segmentation model

In [ ]:
# add model code here ...

# 5. Training

In [ ]:
# add training code here ...

# 4. Evaluation

Let's now run the evaluation over the entire test set!

In [ ]:
# add testing code here ...

# 7. Visualisation

In [ ]:
fig, ax = plt.subplots()

### Your code here...

### Well done, lets now move onto Part 3!
<a target="_blank" href="https://colab.research.google.com/github/Christie-Scientific-Computing/HSST_Workshop/blob/main/Part_3_version_changes.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook 3 In Colab"/>
</a>